# Homework 05: Data Storage

Save a DataFrame as CSV in `data/raw/` and Parquet in `data/processed/`, both paths from `.env`,
reload and validate both, then wrap the pattern in `write_df`/`read_df` utilities that route by
file suffix.

In [1]:
import os, pathlib, datetime as dt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
RAW = pathlib.Path(os.getenv('DATA_DIR_RAW', 'data/raw'))
PROC = pathlib.Path(os.getenv('DATA_DIR_PROCESSED', 'data/processed'))
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)
print('RAW ->', RAW.resolve())
print('PROC ->', PROC.resolve())

def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

RAW -> C:\Users\shaun\OneDrive\Desktop\boot\homework\homework05\data\raw
PROC -> C:\Users\shaun\OneDrive\Desktop\boot\homework\homework05\data\processed


## 1. Sample DataFrame

In [2]:
np.random.seed(7)
dates = pd.date_range('2024-01-01', periods=20, freq='D')
df = pd.DataFrame({
    'date': dates,
    'ticker': ['AAPL'] * 20,
    'price': 150 + np.random.randn(20).cumsum(),
})
df.head()

,date,ticker,price
0,2024-01-01,AAPL,151.690526
1,2024-01-02,AAPL,151.224588
2,2024-01-03,AAPL,151.257408
3,2024-01-04,AAPL,151.664925
4,2024-01-05,AAPL,150.876002


## 2. Save CSV to `data/raw/` and Parquet to `data/processed/`

Timestamped filenames, `DATA_DIR_RAW`/`DATA_DIR_PROCESSED` from `.env`.

In [3]:
csv_path = RAW / f"sample_{ts()}.csv"
df.to_csv(csv_path, index=False)
print('Saved', csv_path)

pq_path = PROC / f"sample_{ts()}.parquet"
try:
    df.to_parquet(pq_path)
    print('Saved', pq_path)
except Exception as e:
    print('Parquet engine not available. Install pyarrow or fastparquet to complete this step.')
    pq_path = None

Saved data\raw\sample_20260827-141109.csv
Saved data\processed\sample_20260827-141109.parquet


## 3. Reload and Validate
Compare shapes and key dtypes against the original DataFrame.

In [4]:
def validate_loaded(original, reloaded):
    return {
        'shape_equal': original.shape == reloaded.shape,
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']) if 'date' in reloaded.columns else False,
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']) if 'price' in reloaded.columns else False,
    }

df_csv = pd.read_csv(csv_path, parse_dates=['date'])
print('CSV reload check:', validate_loaded(df, df_csv))

if pq_path:
    df_pq = pd.read_parquet(pq_path)
    print('Parquet reload check:', validate_loaded(df, df_pq))

CSV reload check: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}
Parquet reload check: {'shape_equal': True, 'date_is_datetime': True, 'price_is_numeric': True}


## 4. Utilities: `write_df` / `read_df`

Route by file suffix, create missing parent directories, and raise a clear error if a Parquet
engine isn't installed instead of a raw traceback.

In [5]:
import typing as t


def detect_format(path: t.Union[str, pathlib.Path]) -> str:
    s = str(path).lower()
    if s.endswith('.csv'):
        return 'csv'
    if s.endswith(('.parquet', '.pq', '.parq')):
        return 'parquet'
    raise ValueError('Unsupported format: ' + s)


def write_df(df: pd.DataFrame, path: t.Union[str, pathlib.Path]) -> pathlib.Path:
    p = pathlib.Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    fmt = detect_format(p)
    if fmt == 'csv':
        df.to_csv(p, index=False)
    else:
        try:
            df.to_parquet(p)
        except ImportError as e:
            raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e
    return p


def read_df(path: t.Union[str, pathlib.Path]) -> pd.DataFrame:
    p = pathlib.Path(path)
    if not p.exists():
        raise FileNotFoundError(f'No file at {p}')
    fmt = detect_format(p)
    if fmt == 'csv':
        header_cols = pd.read_csv(p, nrows=0).columns
        return pd.read_csv(p, parse_dates=['date']) if 'date' in header_cols else pd.read_csv(p)
    try:
        return pd.read_parquet(p)
    except ImportError as e:
        raise RuntimeError('Parquet engine not available. Install pyarrow or fastparquet.') from e


p_csv = RAW / f"util_{ts()}.csv"
p_pq = PROC / f"util_{ts()}.parquet"
write_df(df, p_csv)
write_df(df, p_pq)
print('write_df/read_df round trip, CSV:')
print(read_df(p_csv).head(3))
print('write_df/read_df round trip, Parquet:')
print(read_df(p_pq).head(3))

write_df/read_df round trip, CSV:
        date ticker       price
0 2024-01-01   AAPL  151.690526
1 2024-01-02   AAPL  151.224588
2 2024-01-03   AAPL  151.257408
write_df/read_df round trip, Parquet:
        date ticker       price
0 2024-01-01   AAPL  151.690526
1 2024-01-02   AAPL  151.224588
2 2024-01-03   AAPL  151.257408


In [6]:
# missing-directory and missing-format edge cases
missing_dir_path = RAW / "nested" / "deeper" / f"nested_{ts()}.csv"
write_df(df, missing_dir_path)
print('Missing intermediate directories were created:', missing_dir_path.exists())

try:
    detect_format('sample.txt')
except ValueError as e:
    print('Unsupported suffix raises a clear error:', e)

Missing intermediate directories were created: True
Unsupported suffix raises a clear error: Unsupported format: sample.txt
